In [0]:
import requests
import pandas as pd

print("1. Initializing API parameters for LIT and 12 major connecting hubs...")

# Dictionary of target airports, precise coordinates, and local timezones
airports = {
    "LIT": {"lat": 34.7294, "lon": -92.2242, "tz": "America/Chicago"},
    "ATL": {"lat": 33.6407, "lon": -84.4277, "tz": "America/New_York"},
    "DFW": {"lat": 32.8998, "lon": -97.0403, "tz": "America/Chicago"},
    "ORD": {"lat": 41.9742, "lon": -87.9073, "tz": "America/Chicago"},
    "CLT": {"lat": 35.2140, "lon": -80.9431, "tz": "America/New_York"},
    "IAH": {"lat": 29.9902, "lon": -95.3368, "tz": "America/Chicago"},
    "DEN": {"lat": 39.8617, "lon": -104.6732, "tz": "America/Denver"},
    "DAL": {"lat": 32.8459, "lon": -96.8509, "tz": "America/Chicago"},
    "STL": {"lat": 38.7480, "lon": -90.3700, "tz": "America/Chicago"},
    "LGA": {"lat": 40.7772, "lon": -73.8725, "tz": "America/New_York"},
    "DCA": {"lat": 38.8514, "lon": -77.0377, "tz": "America/New_York"},
    "LAS": {"lat": 36.0803, "lon": -115.1524, "tz": "America/Los_Angeles"},
    "MIA": {"lat": 25.7933, "lon": -80.2906, "tz": "America/New_York"}
}

all_weather_data = []

# Loop through each airport and pull 4 years of data
for airport_code, coords in airports.items():
    print(f"   -> Fetching 4 years of hourly data for {airport_code}...")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": "2022-01-01",
        "end_date": "2025-11-30",
        "hourly": "temperature_2m,precipitation,wind_speed_10m,weather_code",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": coords["tz"]
    }

    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        # Flatten into Pandas DataFrame
        pdf = pd.DataFrame(data['hourly'])
        
        # Add the airport code so we know which city this weather belongs to
        pdf['airport_code'] = airport_code
        
        all_weather_data.append(pdf)
    else:
        print(f"      [!] FAILED for {airport_code}: {response.status_code}")

print("2. Combining all airport data...")
# Concatenate all the individual DataFrames into one massive DataFrame
final_pdf = pd.concat(all_weather_data, ignore_index=True)

print("3. Converting to PySpark DataFrame and writing to Bronze layer...")
sdf = spark.createDataFrame(final_pdf)

# Save as a Bronze Delta Table
sdf.write.format("delta").mode("overwrite").saveAsTable("aviation_project.bronze_historical_weather")

row_count = spark.table("aviation_project.bronze_historical_weather").count()
print(f"4. SUCCESS! Saved {row_count} total hourly weather records to the Bronze layer.")